# Prob distribution
<!-- markdownlint-disable MD013 -->

In [1]:
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

In [3]:
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import beta

from src.constants import *
from src.datasources import glofas

In [4]:
ref = glofas.load_reforecast_frac()
ref["leadtime"] -= 1

In [8]:
ref

,time,leadtime,valid_time,2yr_thresh,5yr_thresh
0,2003-06-01,0,2003-06-02,0.0,0.0
1,2003-06-01,1,2003-06-03,0.0,0.0
2,2003-06-01,2,2003-06-04,0.0,0.0
3,2003-06-01,3,2003-06-05,0.0,0.0
4,2003-06-01,4,2003-06-06,0.0,0.0
...,...,...,...,...,...
68510,2022-11-30,40,2023-01-10,0.0,0.0
68511,2022-11-30,41,2023-01-11,0.0,0.0
68512,2022-11-30,42,2023-01-12,0.0,0.0
68513,2022-11-30,43,2023-01-13,0.0,0.0


In [18]:
max_lt = 10
ref_peaks = (
    ref[ref["leadtime"] <= max_lt]
    .groupby("time")["5yr_thresh"]
    .max()
    .reset_index()
)

In [19]:
ref_peaks["5yr_thresh"].value_counts()

5yr_thresh
0.0    1488
1.0       9
0.1       6
0.5       3
0.6       3
0.3       2
0.2       2
0.7       1
0.8       1
0.4       1
Name: count, dtype: int64

In [28]:
for year in ref_peaks["time"].dt.year.unique():
    print(year)
    dff = ref_peaks[ref_peaks["time"].dt.year == year]
    print(f"actual max {dff['5yr_thresh'].max()}")

    # Assuming your DataFrame is called df and the relevant column is "5yr_thresh"
    data = dff["5yr_thresh"].copy()

    # Adjust values exactly at the boundaries
    data[data <= 0] = 0.0001
    data[data >= 1] = 0.9999
    try:
        # Fit a Beta distribution to your data
        a, b, loc, scale = beta.fit(
            data, floc=0, fscale=1
        )  # Fixing loc=0 and scale=1 for bounded data [0, 1]

        # Calculate the estimated population size based on your sample percentage X
        X = 0.1  # Replace this with your actual sample percentage as a decimal
        population_size = len(data) / X

        # Estimate the expected maximum for the population
        expected_maximum = beta.ppf(
            1 - 1 / population_size, a, b, loc=loc, scale=scale
        )

        # Output the result
        print(f"Expected Maximum for the Population: {expected_maximum}")
    except Exception as e:
        print(e)

2003
actual max 0.8
Expected Maximum for the Population: 0.9195393132245612
2004
actual max 0.0
Solver for the MLE equations failed to converge: The iteration is not making good progress, as measured by the   improvement from the last ten iterations.
2005
actual max 0.0
Solver for the MLE equations failed to converge: The iteration is not making good progress, as measured by the   improvement from the last ten iterations.
2006
actual max 0.0
Solver for the MLE equations failed to converge: The iteration is not making good progress, as measured by the   improvement from the last ten iterations.
2007
actual max 0.0
Solver for the MLE equations failed to converge: The iteration is not making good progress, as measured by the   improvement from the last ten iterations.
2008
actual max 0.0
Solver for the MLE equations failed to converge: The iteration is not making good progress, as measured by the   improvement from the last ten iterations.
2009
actual max 0.0
Solver for the MLE equations 